# Yelp coordinated-fake-review GNN (Colab GPU)

Heterogeneous GNN over the **reviewer ↔ review ↔ business** graph built by `build_yelp_graph.py` (advisor's suggestion: detect coordinated rings via graph structure).

- **Full graph, partial labels:** ~1.03M review nodes (rings intact); labels + train/val/test only on the 178,905 `yelp_split` reviews — the **same held-out test (17,891) as Model B + the text-only baseline**, so results are directly comparable.
- **Features:** review (6) + reviewer (8) + business (3), behavioral/structural (no text in v1 — the GNN's job is the *relational* signal).
- **Task:** semi-supervised node classification on review nodes (fake=1 / genuine=0).

**Setup:** Runtime → Change runtime type → **GPU**. Upload `data/yelp_graph/graph.npz` (a few MB) to the Colab file pane, or mount Drive and set `GRAPH` to its path.

In [ ]:
!pip -q install torch_geometric
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
import numpy as np, torch
from torch_geometric.data import HeteroData

GRAPH = 'graph.npz'   # <- path to the uploaded graph.npz
d = np.load(GRAPH)

def zscore(a):
    a = torch.tensor(a, dtype=torch.float)
    m, s = a.mean(0, keepdim=True), a.std(0, keepdim=True)
    return (a - m) / (s + 1e-6)

data = HeteroData()
data['review'].x   = zscore(d['review_feat'])
data['reviewer'].x = zscore(d['reviewer_feat'])
data['business'].x = zscore(d['business_feat'])

rid = torch.arange(d['review_feat'].shape[0])
ru  = torch.tensor(d['review_user'], dtype=torch.long)
rb  = torch.tensor(d['review_biz'],  dtype=torch.long)
data['review', 'by', 'reviewer'].edge_index   = torch.stack([rid, ru])
data['reviewer', 'wrote', 'review'].edge_index = torch.stack([ru, rid])
data['review', 'on', 'business'].edge_index    = torch.stack([rid, rb])
data['business', 'has', 'review'].edge_index   = torch.stack([rb, rid])

y     = torch.tensor(d['review_label'], dtype=torch.long)
split = torch.tensor(d['review_split'], dtype=torch.long)
data['review'].y = y
data['review'].train_mask = (split == 1)
data['review'].val_mask   = (split == 2)
data['review'].test_mask  = (split == 3)
print(data)
print('train/val/test:', int((split==1).sum()), int((split==2).sum()), int((split==3).sum()))

In [ ]:
import torch.nn.functional as F
from torch_geometric.nn import HeteroConv, SAGEConv, Linear

REL = [('review', 'by', 'reviewer'), ('reviewer', 'wrote', 'review'),
       ('review', 'on', 'business'), ('business', 'has', 'review')]

class HeteroGNN(torch.nn.Module):
    def __init__(self, hidden=128, layers=2, out=2):
        super().__init__()
        self.proj = torch.nn.ModuleDict({nt: Linear(-1, hidden) for nt in ['review','reviewer','business']})
        self.convs = torch.nn.ModuleList([
            HeteroConv({r: SAGEConv((-1, -1), hidden) for r in REL}, aggr='sum')
            for _ in range(layers)])
        self.dropout = torch.nn.Dropout(0.3)
        self.out = Linear(hidden, out)
    def forward(self, x_dict, eidx):
        x = {k: self.proj[k](v).relu() for k, v in x_dict.items()}
        for conv in self.convs:
            x = {k: self.dropout(v.relu()) for k, v in conv(x, eidx).items()}
        return self.out(x['review'])

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data = data.to(dev)
model = HeteroGNN(hidden=128, layers=2).to(dev)
with torch.no_grad():            # init lazy (-1) Linear modules before building the optimizer
    model(data.x_dict, data.edge_index_dict)
opt = torch.optim.Adam(model.parameters(), lr=5e-3, weight_decay=5e-4)

tr, va, te = data['review'].train_mask, data['review'].val_mask, data['review'].test_mask
yb = data['review'].y
ytr = yb[tr].cpu().numpy()
cw = torch.tensor([len(ytr)/(2*(ytr==0).sum()), len(ytr)/(2*(ytr==1).sum())], dtype=torch.float, device=dev)

def evaluate(mask):
    model.eval()
    with torch.no_grad():
        pred = model(data.x_dict, data.edge_index_dict).argmax(1)
    yt = yb[mask].cpu().numpy(); yp = pred[mask].cpu().numpy()
    return (f1_score(yt, yp, average='macro', zero_division=0),
            recall_score(yt, yp, pos_label=1, zero_division=0),
            precision_score(yt, yp, pos_label=1, zero_division=0),
            accuracy_score(yt, yp))

best_val, best_state, patience, bad = -1.0, None, 15, 0
for epoch in range(1, 151):
    model.train(); opt.zero_grad()
    out = model(data.x_dict, data.edge_index_dict)
    loss = F.cross_entropy(out[tr], yb[tr], weight=cw)
    loss.backward(); opt.step()
    vmf1 = evaluate(va)[0]
    if vmf1 > best_val:
        best_val = vmf1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        bad = 0
    else:
        bad += 1
    if epoch % 5 == 0 or bad == 0:
        print(f'epoch {epoch:3d}  loss {loss.item():.4f}  val macroF1 {vmf1:.4f}  (best {best_val:.4f})')
    if bad >= patience:
        print(f'early stop @ epoch {epoch}'); break

model.load_state_dict(best_state)
mf1, fr, fp, acc = evaluate(te)
print(f'\nTEST (n={int(te.sum())}, SAME held-out reviews as Model B + baseline):')
print(f'  macro-F1 {mf1:.3%}  |  fake recall {fr:.3%}  fake precision {fp:.3%}  |  accuracy {acc:.3%}')
print('  Compare to Model B / text-only baseline via evaluate_yelp_modelb_vs_baseline.py')
torch.save({'state_dict': best_state, 'val_macro_f1': best_val,
            'test': {'macro_f1': mf1, 'fake_recall': fr, 'fake_precision': fp, 'acc': acc}},
           'yelp_gnn.pt')
print('saved yelp_gnn.pt')